# 02 — Confidence controller and prequential runner

Refs #20/#21/#39. Inspect executed decisions and chronology, then demonstrate a small CPU restore using the existing runner.

## Environment and artifact setup

Use Python 3.12 as documented in the repository. Install the pinned PyTorch requirement (requirements-model.txt from the CPU index, or requirements-cuda.txt from the cu126 index), plus requirements-dev.txt. Notebook presentation dependencies used for recorded validation: nbformat 5.10.4, nbclient 0.10.2, ipykernel 6.29.5, pandas 3.0.1, matplotlib 3.10.8.

Open from the repository or set HUNG_REPO_ROOT. Download/extract identified inputs and run artifacts; set HUNG_INPUT_BASE and HUNG_RUN_ROOT if locations differ. Missing files fail explicitly. Large inputs/checkpoints stay outside Git; only load trusted project checkpoints.

Default cells inspect existing evidence and run small CPU examples. They do not launch matrix training. Recorded matrix code SHA: a34a49e5f04920ef30456fe6c0ea56eb2f1d659a, diagnostic config v1 and selection v2. Clean CPU reproduction was validated in a new isolated repo-local Python 3.13.7 environment with torch 2.8.0+cpu, pip check, the full unit suite and a fresh kernel per notebook. README's reference Python 3.12 environment was not executed in this local check. The global CUDA environment has an unrelated ultralytics/numpy conflict; it was not inherited by the isolated environment.


In [1]:
import hashlib
import json
import os
import sys
from pathlib import Path

import pandas as pd

candidates = (
    [Path(os.environ["HUNG_REPO_ROOT"])]
    if os.environ.get("HUNG_REPO_ROOT")
    else [Path.cwd(), *Path.cwd().parents]
)
REPO = next(
    (
        path.resolve()
        for path in candidates
        if (path / "experiments/hung_sprint2/model.py").is_file()
    ),
    None,
)
if REPO is None:
    raise FileNotFoundError("Set HUNG_REPO_ROOT to the checked-out repository.")
sys.path.insert(0, str(REPO))
RUN_ROOT = Path(
    os.environ.get("HUNG_RUN_ROOT", str(REPO / "runs/hung-39-representative-local-20261025T2103"))
).resolve()
INPUT_BASE = Path(
    os.environ.get("HUNG_INPUT_BASE", str(REPO / "data/inputs/issue18-input-handoff-v1-20261019"))
).resolve()


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


identity = read_json(RUN_ROOT / "matrix_identity.json")
config = read_json(REPO / "experiments/hung_sprint2/development_config_v1.json")
assert identity["phase"] == "representative"
assert identity["config_sha256"] == sha256(
    REPO / "experiments/hung_sprint2/development_config_v1.json"
)
assert identity["selection_sha256"] == sha256(
    REPO / "docs/reports/hung-development-selection-20261025-v2.json"
)
print(
    {
        "run_id": RUN_ROOT.name,
        "execution_code_sha": identity["code"]["commit"],
        "device": identity["device"],
    }
)

{'run_id': 'hung-39-representative-local-20261025T2103', 'execution_code_sha': 'a34a49e5f04920ef30456fe6c0ea56eb2f1d659a', 'device': 'cuda'}


## Decision rule and event order

Daily paired improvement is \(\Delta_t=MAE_{active,t}-MAE_{challenger,t}\). The controller resamples contiguous segments of complete valid observations, computes a one-sided percentile lower bound, applies Bonferroni alpha/2 to two challengers, and switches only for a bound strictly above delta. Diagnostic v1: levels {0,.1,.2}, 14 scheduled days, Bmin=5, segment=2, 100 resamples, alpha=.05, delta=0. Gaps are compressed in the valid-observation bootstrap sequence; this policy remains for team review.

Order: fork equal candidate state → persist forecasts → accept complete arrived labels → score/update → decide for future forecasts → carry the trained selected state. Old-policy audit costs are separate; its output cannot influence selection. See controller.py, runner.py, replay_inputs.py and checkpoint.py.

In [2]:
from dataclasses import asdict

from experiments.hung_sprint2.controller import ControllerConfig, DailyScore, decide_block

controller = ControllerConfig(
    levels=(0.0, config["replay"]["level_one"], config["replay"]["level_two"]),
    initial_level=config["replay"]["initial_level"],
    block_days=config["replay"]["block_days"],
    min_complete_days=config["replay"]["min_complete_days"],
    delta=config["replay"]["delta"],
    alpha=config["replay"]["alpha"],
    bootstrap_segment_days=config["replay"]["bootstrap_segment_days"],
    bootstrap_resamples=config["replay"]["bootstrap_resamples"],
    seed=7,
    challenger_rule=config["replay"]["challenger_rule"],
)
days = [DailyScore(f"fixture-{index}", True, True, (2.0, 1.0, 1.5)) for index in range(5)]
decision = decide_block(controller, current_level=0.0, block_id=0, days=days)
assert decision.selected_level == 0.1
assert decision == decide_block(controller, current_level=0.0, block_id=0, days=days)
print("Synthetic expected-value example:", asdict(decision))
real_manifest = read_json(RUN_ROOT / "14d/seed-7/entity-1153/replay-90/run_manifest.json")
pd.DataFrame(real_manifest["event_evidence"]["decisions"])

Synthetic expected-value example: {'current_level': 0.0, 'selected_level': 0.1, 'valid_days': 5, 'skipped_days': 0, 'lower_bounds': (None, 1.0, 0.5), 'alpha_per_challenger': 0.025, 'reason': 'switch'}


,alpha_per_challenger,current_level,lower_bounds,reason,selected_level,skipped_days,valid_days
0,0.025,0.0,"[None, -0.00025554852826254707, -0.00045024710...",no_qualifying_challenger,0.0,0,14
1,0.025,0.0,"[None, -0.0002939536103180477, -0.000531250344...",no_qualifying_challenger,0.0,0,14
2,0.025,0.0,"[None, 0.00040032794432980675, 0.0008812272655...",switch,0.2,0,14
3,0.025,0.2,"[-0.0009801305830478668, -0.000399547442793846...",no_qualifying_challenger,0.2,4,10
4,0.025,0.2,"[None, None, None]",insufficient_days,0.2,14,0
5,0.025,0.2,"[None, None, None]",insufficient_days,0.2,14,0


In [3]:
pd.DataFrame({"event": real_manifest["event_evidence"]["events"][:18]})

,event
0,fork:0
1,forecast:2022-12-01T00:00:00+08:00
2,unscored:2022-12-01T00:00:00+08:00
3,score:2022-12-01T00:00:00+08:00
4,update:2022-12-01T00:00:00+08:00
5,forecast:2022-12-02T00:00:00+08:00
6,unscored:2022-12-02T00:00:00+08:00
7,score:2022-12-02T00:00:00+08:00
8,update:2022-12-02T00:00:00+08:00
9,forecast:2022-12-03T00:00:00+08:00


## CPU chronology and restore example

Use the first real Development forecast/complete label for entity 2264 in a temporary CPU demonstration. This does not resume CUDA runner RNG/state on CPU or write into the matrix stages.

In [4]:
import tempfile

import torch

from ev_inputs.loader import iter_forecasts
from experiments.hung_sprint2.checkpoint import load_runner, save_runner
from experiments.hung_sprint2.runner import LabelArrival, Origin, PrequentialRunner
from experiments.hung_sprint2.training import initialize_target, load_checkpoint

torch.set_num_threads(1)
base = RUN_ROOT / "7d/seed-7"
source_manifest = read_json(base / "source/run_manifest.json")
target_manifest = read_json(base / "entity-2264/target/run_manifest.json")
source, _, latent_scale, _ = load_checkpoint(
    base / "source/source_checkpoint.pt",
    expected_binding=source_manifest["input_binding"],
    map_location="cpu",
)
model, optimizer, _, _ = load_checkpoint(
    base / "entity-2264/target/target_checkpoint.pt",
    expected_binding=target_manifest["input_binding"],
    map_location="cpu",
)
_, reference, _ = initialize_target(source, learning_rate=config["target"]["learning_rate"])
input_run = INPUT_BASE / "urbanev-SZH-7d"
record = next(
    row
    for row in iter_forecasts(
        input_run, expected_manifest_sha256=identity["input_manifest_sha256"]["7"]
    )
    if row["role"] == "development" and row["identity"][2] == "2264"
)
with (input_run / "labels.jsonl").open(encoding="utf-8") as stream:
    label = next(
        row
        for row in map(json.loads, stream)
        if row["identity"][2] == "2264" and row["origin"] == record["origin"]
    )
assert all(label["file_valid"]) and all(value is not None for value in label["energy_kwh"])
replay = PrequentialRunner(
    model,
    optimizer,
    reference,
    latent_scale,
    binding=target_manifest["input_binding"],
    controller=controller,
    epsilon=config["target"]["epsilon"],
    energy_scale=record["energy_scale"],
)
replay.forecast(
    Origin(
        record["origin"],
        0,
        torch.tensor(record["x_hist"], dtype=torch.float32).unsqueeze(0),
        torch.tensor(record["c_future"], dtype=torch.float32).unsqueeze(0),
    )
)
assert replay.issued[0]["normalized_mae"] is None
replay.arrive(
    LabelArrival(
        record["origin"],
        True,
        torch.tensor(label["energy_kwh"], dtype=torch.float32) / record["energy_scale"],
        torch.ones(24, dtype=torch.bool),
    )
)
temporary_root = REPO / ".cache/notebook-validation"
temporary_root.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryDirectory(dir=temporary_root) as directory:
    checkpoint = Path(directory) / "runner.pt"
    save_runner(checkpoint, replay)
    restored = load_runner(
        checkpoint, expected_binding=target_manifest["input_binding"], device="cpu"
    )
    assert restored.events == replay.events
    assert restored.next_index == replay.next_index
print(
    {
        "events": replay.events,
        "candidate_updates": replay.update_steps,
        "restored_cursor": restored.next_index,
        "actual_device": str(restored.device),
    }
)
pd.DataFrame(real_manifest["runtime_costs"]["candidate"])

{'events': ['fork:0', 'forecast:2022-12-01T00:00:00+08:00', 'score:2022-12-01T00:00:00+08:00', 'update:2022-12-01T00:00:00+08:00'], 'candidate_updates': [1, 1, 1], 'restored_cursor': 1, 'actual_device': 'cpu'}


,inference_calls,inference_ns,update_ns,update_steps
0,59,146700700,528693100,52
1,59,45421200,415791200,52
2,59,43416100,382025900,52


## Limits

This checks one CPU event and restore; comprehensive boundary/missing/keep/switch/audit fixtures and executed 42→90 CUDA replay remain the stronger evidence in phase reports. Independent implementation agreement and statistical adequacy remain open.